In [28]:
import pandas as pd
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
import joblib

In [29]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.freq_maps = {}

    def fit(self, X, y=None):
        for col in X.columns:
            self.freq_maps[col] = X[col].value_counts(normalize=True).to_dict()
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col in X_copy.columns:
            X_copy[col] = X_copy[col].map(self.freq_maps[col]).fillna(0)
        return X_copy

In [30]:
class DateFeatureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = pd.DataFrame(X).copy()
        X_copy['trans_date_trans_time'] = pd.to_datetime(X_copy['trans_date_trans_time'])
        X_copy['dob'] = pd.to_datetime(X_copy['dob'])

        hours = X_copy['trans_date_trans_time'].dt.hour
        is_night = ((hours >= 22) | (hours < 4)).astype(int)

        weekday = X_copy['trans_date_trans_time'].dt.weekday
        
        age = (X_copy['trans_date_trans_time'] - X_copy['dob']).dt.days // 365

        return pd.DataFrame({
            'is_night': is_night,
            'weekday': weekday,
            'age': age
        })

In [31]:
cols_oneHot = ['category', 'gender']  
cols_freq    = ['merchant', 'city', 'state', 'job']
cols_dates = ['trans_date_trans_time', 'dob']
cols_num     = ['amt', 'city_pop']


In [32]:
date_processing_pipeline = Pipeline(steps=[
    ('extract', DateFeatureExtractor()),  
    ('scale', StandardScaler())           
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), cols_num),
        ('cat_oneHot', OneHotEncoder(handle_unknown='ignore'), cols_oneHot),
        ('cat_freq', FrequencyEncoder(), cols_freq),
        ('date_branch', date_processing_pipeline, cols_dates),
    ],
    remainder='drop'
)

In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.ensemble import VotingClassifier
from xgboost import XGBClassifier

clf1 = LogisticRegression()
clf2 = RandomForestClassifier(n_estimators=100, max_depth=10, n_jobs=-1)
clf3 = xgb_model = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, n_jobs=-1) 

ensemble_model = VotingClassifier(
    estimators=[('lr', clf1), ('rf', clf2), ('xgb', clf3)],
    voting='soft', 
    weights=[1, 5, 3]
)

In [34]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', ensemble_model)
])

In [35]:
data = pd.read_csv('./../data/balanced_data.csv')

X_train = data.drop(columns='is_fraud')
Y_train = data['is_fraud']

In [36]:
pipeline.fit(X_train, Y_train)

joblib.dump(pipeline, 'model_ensemble.joblib')

['model_ensemble.joblib']